In [1]:
# ===== HEADER: loader + helpers =====
import time, itertools, random
import pandas as pd
import numpy as np

def load_ctx_drug(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, index_col=0)
    df.columns = df.columns.str.strip()
    df.index = df.index.astype(str).str.strip()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def pairwise_tissue_sets(df: pd.DataFrame):
    """Return drugs, tissues, and S[a][b] = set of tissues where a>b."""
    drugs = list(df.columns)
    tissues = list(df.index)
    n = len(drugs)
    S = [[set() for _ in range(n)] for __ in range(n)]
    for i, a in enumerate(drugs):
        for j, b in enumerate(drugs):
            if i==j: continue
            wins = set()
            for t in tissues:
                va, vb = df.at[t, a], df.at[t, b]
                if pd.isna(va) or pd.isna(vb):
                    continue
                if va > vb:
                    wins.add(t)
            S[i][j] = wins
    return drugs, tissues, S

def path_intersection_prefix(order_idxs, S, lam):
    """Given index order and S[i][j], return the longest λ-consistent prefix path and its common tissues S*."""
    if not order_idxs: return [], set(), []
    S_common = None
    adj_sets = []
    path = [order_idxs[0]]
    for i,j in zip(order_idxs[:-1], order_idxs[1:]):
        Sab = S[i][j]
        S_common = Sab if S_common is None else (S_common & Sab)
        if len(S_common) >= lam:
            path.append(j)
            adj_sets.append(Sab & S_common)
        else:
            break
    return path, (S_common if S_common is not None else set()), adj_sets

def print_result(tag, elapsed_s, order, S):
    print(f"[{tag}] time = {elapsed_s:.3f} s | path length = {len(order)} | |S| = {len(S)}")
    if order:
        print("  Path:", " > ".join(order))
    else:
        print("  Path: (empty)")


In [2]:
# ===== TA-RP (Ranked Pairs, chain λ-consistent path) =====
def ta_ranked_pairs_path(df: pd.DataFrame, lam: int):
    drugs, tissues, S = pairwise_tissue_sets(df)
    n = len(drugs)

    # Sort directed pairs by strength (|S[i][j]| desc), break ties deterministically
    pairs = [(i,j,len(S[i][j])) for i in range(n) for j in range(n) if i!=j and len(S[i][j])>0]
    pairs.sort(key=lambda x:(x[2], -x[0], -x[1]), reverse=True)

    # Lock edges while maintaining chain λ-consistency along any path (greedy)
    # We’ll keep a DAG G and then extract its longest path (λ-checked).
    G = [[False]*n for _ in range(n)]

    def creates_cycle(u,v):
        # check if adding u->v creates cycle via DFS
        stack=[v]; seen=set()
        while stack:
            x=stack.pop()
            if x==u: return True
            for y in range(n):
                if G[x][y] and y not in seen:
                    seen.add(y); stack.append(y)
        return False

    for i,j,_w in pairs:
        if creates_cycle(i,j):
            continue
        # Tentatively add and ensure there exists a λ-consistent path between i and j through G
        # We enforce per-edge intersection along the path (like CDR on this subgraph)
        G[i][j] = True
        # quick check: if this single edge itself has at least lam, keep; else revert
        if len(S[i][j]) < lam:
            G[i][j] = False
            continue

    # Extract longest λ-consistent path from the DAG using the S sets
    # Try all starts with DFS, propagate intersection
    best_path_idx, best_S = [], set()
    for start in range(n):
        stack=[(start,[start], None)]
        while stack:
            cur, path, S_common = stack.pop()
            for nxt in range(n):
                if not G[cur][nxt] or nxt in path:
                    continue
                Sab = S[cur][nxt]
                new_S = Sab if S_common is None else (S_common & Sab)
                if len(new_S) >= lam:
                    new_path = path + [nxt]
                    stack.append((nxt, new_path, new_S))
                    if len(new_path) > len(best_path_idx):
                        best_path_idx, best_S = new_path, new_S

    return [drugs[i] for i in best_path_idx], best_S, []

def run_tarp(csv_path, lam=30):
    df = load_ctx_drug(csv_path)
    t0 = time.time()
    order, S_common, _ = ta_ranked_pairs_path(df, lam)
    elapsed = time.time() - t0
    print_result("TA-RP-PATH", elapsed, order, S_common)


In [3]:
# ============ TA-RP (chain-λ) with SOFT DEADLINE + Benchmarks ============

def ta_ranked_pairs_path_soft(df: pd.DataFrame, lam: int, time_budget_s: float = 3600.0):
    """
    Soft-deadline version of your ta_ranked_pairs_path:
    - Periodically checks wall time during edge locking and path search.
    - Returns best-so-far path if budget is exceeded.
    """
    t0 = time.time()
    drugs, tissues, S = pairwise_tissue_sets(df)
    n = len(drugs)

    # Sort directed pairs by strength (|S[i][j]| desc), deterministic tie-break
    pairs = [(i, j, len(S[i][j])) for i in range(n) for j in range(n)
             if i != j and len(S[i][j]) > 0]
    pairs.sort(key=lambda x: (x[2], -x[0], -x[1]), reverse=True)

    # Build a DAG by locking edges that don't create cycles AND have per-edge support ≥ λ
    G = [[False]*n for _ in range(n)]

    def creates_cycle(u, v):
        stack = [v]; seen = set()
        while stack:
            # soft deadline check inside DFS
            if time.time() - t0 > time_budget_s:
                return False  # return gracefully; we’ll mark deadline later
            x = stack.pop()
            if x == u:
                return True
            if x in seen:
                continue
            seen.add(x)
            for y in range(n):
                if G[x][y] and y not in seen:
                    stack.append(y)
        return False

    deadline_hit = False

    for i, j, w in pairs:
        if time.time() - t0 > time_budget_s:
            deadline_hit = True
            break
        if w < lam:
            continue
        if creates_cycle(i, j):
            continue
        G[i][j] = True

    # Extract the longest λ-consistent path from the locked DAG (DFS with soft deadline)
    best_path_idx, best_S = [], set()
    for start in range(n):
        if time.time() - t0 > time_budget_s:
            deadline_hit = True
            break
        stack = [(start, [start], None)]   # (cur, path, S_common)
        while stack:
            if time.time() - t0 > time_budget_s:
                deadline_hit = True
                break
            cur, path, S_common = stack.pop()
            for nxt in range(n):
                if not G[cur][nxt] or nxt in path:
                    continue
                Sab = S[cur][nxt]
                new_S = Sab if S_common is None else (S_common & Sab)
                if len(new_S) >= lam:
                    new_path = path + [nxt]
                    stack.append((nxt, new_path, new_S))
                    if len(new_path) > len(best_path_idx):
                        best_path_idx, best_S = new_path, new_S
        if deadline_hit:
            break

    elapsed = time.time() - t0
    order = [drugs[i] for i in best_path_idx]
    status = "deadline" if deadline_hit else "ok"
    return order, best_S, elapsed, status


# ---------------- Benchmark 1: λ-increasing on ALL drugs ----------------
def tarp_lambda_increasing(csv_path="efficacy.csv",
                           lam_start=5, lam_end=60, lam_step=5,
                           time_budget_s=3600,
                           out_xlsx="tarp_lambda_increasing.xlsx"):
    """
    Uses ALL drugs from the CSV. For each λ, runs TA-RP-PATH with a soft deadline.
    Writes Excel with: method, n_drugs, lambda, path, path_length, exec_time_s, status
    """
    df_full = load_ctx_drug(csv_path)
    n_drugs = df_full.shape[1]
    rows = []
    print(f"TA-RP-PATH λ-sweep on ALL {n_drugs} drugs (soft deadline per λ = {time_budget_s//60:.0f} min)")

    for lam in range(lam_start, lam_end + 1, lam_step):
        order, S_common, elapsed, status = ta_ranked_pairs_path_soft(df_full, lam, time_budget_s=time_budget_s)
        rows.append({
            "method": "TA-RP-PATH",
            "n_drugs": n_drugs,
            "lambda": lam,
            "path": " > ".join(order),
            "path_length": len(order),
            "exec_time_s": elapsed,
            "status": status
        })
        print(f"  λ={lam:3d} | status={status:8s} | time={elapsed:.2f} s | len={len(order)}")
        if status == "deadline":
            print("  ⏱️ Soft deadline hit; stopping λ sweep.")
            break

    out_df = pd.DataFrame(rows)
    with pd.ExcelWriter(out_xlsx) as xw:
        out_df.to_excel(xw, sheet_name="results", index=False)
    print(f"✅ Saved -> {out_xlsx}")
    return out_df


# ------------- Benchmark 2: drugs-increasing (fixed λ), nested -------------
def tarp_drug_increasing(csv_path="efficacy.csv",
                         lam_fixed=30,
                         start_size=5, step=5, seed=42,
                         time_budget_s=3600,
                         out_xlsx="tarp_drug_increasing.xlsx"):
    """
    Grows the number of drugs with a fixed λ. Nested, reproducible subsets (same seed).
    Soft deadline per size. Stops when deadline hits or when all drugs are tested.
    """
    df_full = load_ctx_drug(csv_path)
    n_total = df_full.shape[1]
    sizes = list(range(start_size, n_total + 1, step))
    rows = []

    # reproducible nested subset of columns
    rng = random.Random(seed)
    cols = list(df_full.columns)
    rng.shuffle(cols)

    print(f"TA-RP-PATH drug-sweep at λ={lam_fixed}; total drugs = {n_total} (soft deadline per size = {time_budget_s//60:.0f} min)")
    for n in sizes:
        sub = df_full[cols[:n]]
        order, S_common, elapsed, status = ta_ranked_pairs_path_soft(sub, lam_fixed, time_budget_s=time_budget_s)
        rows.append({
            "method": "TA-RP-PATH",
            "n_drugs": n,
            "lambda": lam_fixed,
            "path": " > ".join(order),
            "path_length": len(order),
            "exec_time_s": elapsed,
            "status": status
        })
        print(f"  n={n:3d} | status={status:8s} | time={elapsed:.2f} s | len={len(order)}")
        if status == "deadline":
            print("  ⏱️ Soft deadline hit; stopping size growth.")
            break

    out_df = pd.DataFrame(rows)
    with pd.ExcelWriter(out_xlsx) as xw:
        out_df.to_excel(xw, sheet_name="results", index=False)
    print(f"✅ Saved -> {out_xlsx}")
    return out_df


# ---------------------- Example calls (uncomment to run) ----------------------
# tarp_lambda_increasing("efficacy.csv",
#                        lam_start=3, lam_end=60, lam_step=3,
#                        time_budget_s=3600,
#                        out_xlsx="tarp_lambda_increasing.xlsx")

# tarp_drug_increasing("efficacy.csv",
#                      lam_fixed=30,
#                      start_size=5, step=5, seed=42,
#                      time_budget_s=3600,
#                      out_xlsx="tarp_drug_increasing.xlsx")


In [ ]:
 tarp_drug_increasing("efficacy.csv",
                      lam_fixed=30,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m.xlsx")

TA-RP-PATH drug-sweep at λ=30; total drugs = 273 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.03 s | len=7
  n= 15 | status=ok       | time=0.07 s | len=9
  n= 20 | status=ok       | time=0.16 s | len=10
  n= 25 | status=ok       | time=0.41 s | len=11
  n= 30 | status=ok       | time=0.85 s | len=11
  n= 35 | status=ok       | time=2.87 s | len=12
  n= 40 | status=ok       | time=14.18 s | len=13
  n= 45 | status=ok       | time=46.22 s | len=13
  n= 50 | status=ok       | time=114.95 s | len=13
  n= 55 | status=ok       | time=254.03 s | len=13
  n= 60 | status=ok       | time=508.16 s | len=14
  n= 65 | status=ok       | time=1041.00 s | len=14
  n= 70 | status=ok       | time=1419.95 s | len=14
  n= 75 | status=deadline | time=1800.00 s | len=14
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_increasing30m.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,30,VEMURAFENIB > NITAZOXANIDE > PACLITAXEL > ILOP...,4,0.006306,ok
1,TA-RP-PATH,10,30,THEOPHYLLINE > DEXAMETHASONE > ILORASERTIB > Z...,7,0.026577,ok
2,TA-RP-PATH,15,30,THEOPHYLLINE > DEXAMETHASONE > ILORASERTIB > T...,9,0.069611,ok
3,TA-RP-PATH,20,30,THEOPHYLLINE > DEXAMETHASONE > FENRETINIDE > H...,10,0.158157,ok
4,TA-RP-PATH,25,30,THEOPHYLLINE > DEXAMETHASONE > IRINOTECAN HYDR...,11,0.412742,ok
5,TA-RP-PATH,30,30,THEOPHYLLINE > DEXAMETHASONE > FENRETINIDE > P...,11,0.852746,ok
6,TA-RP-PATH,35,30,THEOPHYLLINE > DEXAMETHASONE > IRINOTECAN HYDR...,12,2.865228,ok
7,TA-RP-PATH,40,30,THEOPHYLLINE > DEXAMETHASONE > CURCUMIN > MECH...,13,14.183151,ok
8,TA-RP-PATH,45,30,THEOPHYLLINE > ITRACONAZOLE > IRINOTECAN HYDRO...,13,46.221761,ok
9,TA-RP-PATH,50,30,THEOPHYLLINE > SANGUINARINE SULFATE > CURCUMIN...,13,114.954337,ok


In [ ]:
 tarp_lambda_increasing("efficacy.csv",
                        lam_start=5, lam_end=49, lam_step=5,
                        time_budget_s=120,
                        out_xlsx="tarp_lambda_increasing.xlsx")

TA-RP-PATH λ-sweep on ALL 273 drugs (soft deadline per λ = 2 min)
  λ=  5 | status=deadline | time=120.00 s | len=17
  ⏱️ Soft deadline hit; stopping λ sweep.
✅ Saved -> tarp_lambda_increasing.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,273,5,ASPIRIN > BUMETANIDE > NITAZOXANIDE > METERGOL...,17,120.000003,deadline


In [ ]:
tarp_drug_increasing("efficacy.csv",
                      lam_fixed=35,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasingL3530m.xlsx")

TA-RP-PATH drug-sweep at λ=35; total drugs = 273 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.03 s | len=6
  n= 15 | status=ok       | time=0.08 s | len=8
  n= 20 | status=ok       | time=0.17 s | len=9
  n= 25 | status=ok       | time=0.37 s | len=10
  n= 30 | status=ok       | time=0.68 s | len=10
  n= 35 | status=ok       | time=1.96 s | len=11
  n= 40 | status=ok       | time=8.07 s | len=12
  n= 45 | status=ok       | time=23.76 s | len=12
  n= 50 | status=ok       | time=52.76 s | len=12
  n= 55 | status=ok       | time=105.20 s | len=12
  n= 60 | status=ok       | time=197.16 s | len=13
  n= 65 | status=ok       | time=369.90 s | len=13
  n= 70 | status=ok       | time=485.53 s | len=13
  n= 75 | status=ok       | time=771.95 s | len=13
  n= 80 | status=ok       | time=1265.47 s | len=13
  n= 85 | status=deadline | time=1800.00 s | len=13
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_inc

,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,35,VEMURAFENIB > NITAZOXANIDE > PACLITAXEL > ILOP...,4,0.007793,ok
1,TA-RP-PATH,10,35,THEOPHYLLINE > DEXAMETHASONE > ILORASERTIB > Z...,6,0.031623,ok
2,TA-RP-PATH,15,35,PD-0166285 > MESALAMINE > TRAZODONE > ZIPRASID...,8,0.081380,ok
3,TA-RP-PATH,20,35,DEXAMETHASONE > IRINOTECAN HYDROCHLORIDE > MES...,9,0.174946,ok
4,TA-RP-PATH,25,35,THEOPHYLLINE > DEXAMETHASONE > FENRETINIDE > H...,10,0.374066,ok
5,TA-RP-PATH,30,35,DEXAMETHASONE > IRINOTECAN HYDROCHLORIDE > MEC...,10,0.679496,ok
6,TA-RP-PATH,35,35,THEOPHYLLINE > ITRACONAZOLE > IRINOTECAN HYDRO...,11,1.955492,ok
7,TA-RP-PATH,40,35,THEOPHYLLINE > DEXAMETHASONE > CURCUMIN > ARIP...,12,8.071682,ok
8,TA-RP-PATH,45,35,THEOPHYLLINE > DEXAMETHASONE > STREPTOZOCIN > ...,12,23.761290,ok
9,TA-RP-PATH,50,35,THEOPHYLLINE > HALOPERIDOL DECANOATE > MECHLOR...,12,52.764649,ok


In [ ]:
tarp_drug_increasing("efficacy.csv",
                      lam_fixed=25,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasingL2530m.xlsx")

TA-RP-PATH drug-sweep at λ=25; total drugs = 273 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.03 s | len=7
  n= 15 | status=ok       | time=0.09 s | len=10
  n= 20 | status=ok       | time=0.25 s | len=11
  n= 25 | status=ok       | time=0.92 s | len=11
  n= 30 | status=ok       | time=2.27 s | len=11
  n= 35 | status=ok       | time=9.16 s | len=12
  n= 40 | status=ok       | time=53.72 s | len=13
  n= 45 | status=ok       | time=193.29 s | len=14
  n= 50 | status=ok       | time=525.42 s | len=14
  n= 55 | status=ok       | time=1249.60 s | len=14
  n= 60 | status=deadline | time=1800.00 s | len=14
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_increasingL2530m.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,25,VEMURAFENIB > NITAZOXANIDE > PACLITAXEL > ILOP...,4,0.007565,ok
1,TA-RP-PATH,10,25,THEOPHYLLINE > DEXAMETHASONE > ILORASERTIB > Z...,7,0.033380,ok
2,TA-RP-PATH,15,25,THEOPHYLLINE > DEXAMETHASONE > PD-0166285 > IL...,10,0.093779,ok
3,TA-RP-PATH,20,25,THEOPHYLLINE > DEXAMETHASONE > IRINOTECAN HYDR...,11,0.250869,ok
4,TA-RP-PATH,25,25,THEOPHYLLINE > DEXAMETHASONE > FENRETINIDE > L...,11,0.915961,ok
5,TA-RP-PATH,30,25,DEXAMETHASONE > IRINOTECAN HYDROCHLORIDE > MEC...,11,2.269444,ok
6,TA-RP-PATH,35,25,THEOPHYLLINE > ITRACONAZOLE > IRINOTECAN HYDRO...,12,9.164600,ok
7,TA-RP-PATH,40,25,THEOPHYLLINE > ITRACONAZOLE > CURCUMIN > PF-04...,13,53.718001,ok
8,TA-RP-PATH,45,25,THEOPHYLLINE > DEXAMETHASONE > CURCUMIN > MECH...,14,193.287342,ok
9,TA-RP-PATH,50,25,THEOPHYLLINE > SANGUINARINE SULFATE > CURCUMIN...,14,525.419347,ok


In [ ]:
tarp_drug_increasing("efficacy-diabities.csv",
                      lam_fixed=30,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_diabities.xlsx")

TA-RP-PATH drug-sweep at λ=30; total drugs = 273 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.02 s | len=5
  n= 10 | status=ok       | time=0.07 s | len=7
  n= 15 | status=ok       | time=0.18 s | len=7
  n= 20 | status=ok       | time=0.35 s | len=9
  n= 25 | status=ok       | time=1.03 s | len=10
  n= 30 | status=ok       | time=1.65 s | len=10
  n= 35 | status=ok       | time=4.18 s | len=11
  n= 40 | status=ok       | time=23.43 s | len=12
  n= 45 | status=ok       | time=108.90 s | len=13
  n= 50 | status=ok       | time=249.11 s | len=13
  n= 55 | status=ok       | time=449.01 s | len=13
  n= 60 | status=ok       | time=928.44 s | len=13
  n= 65 | status=ok       | time=1639.16 s | len=13
  n= 70 | status=deadline | time=1800.00 s | len=13
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_increasing30m_diabities.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,30,CARBOPLATIN > CHEMBL:CHEMBL493863 > LEUCOVORIN...,5,0.016214,ok
1,TA-RP-PATH,10,30,DACTINOMYCIN > WORTMANNIN > RITUXIMAB > LEUCOV...,7,0.073028,ok
2,TA-RP-PATH,15,30,DACTINOMYCIN > MIDAZOLAM HYDROCHLORIDE > RITUX...,7,0.177297,ok
3,TA-RP-PATH,20,30,SULFASALAZINE > CYCLOPHOSPHAMIDE ANHYDROUS > D...,9,0.354157,ok
4,TA-RP-PATH,25,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > PREGNANT...,10,1.030681,ok
5,TA-RP-PATH,30,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > MK-2461 ...,10,1.653115,ok
6,TA-RP-PATH,35,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,11,4.183852,ok
7,TA-RP-PATH,40,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SODIUM ...,12,23.432402,ok
8,TA-RP-PATH,45,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,13,108.897907,ok
9,TA-RP-PATH,50,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,13,249.114028,ok


In [ ]:
tarp_drug_increasing("efficacy-diabities.csv",
                      lam_fixed=35,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_diabities_l35.xlsx")

TA-RP-PATH drug-sweep at λ=35; total drugs = 273 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.04 s | len=7
  n= 15 | status=ok       | time=0.09 s | len=7
  n= 20 | status=ok       | time=0.19 s | len=8
  n= 25 | status=ok       | time=0.42 s | len=9
  n= 30 | status=ok       | time=0.86 s | len=9
  n= 35 | status=ok       | time=1.75 s | len=10
  n= 40 | status=ok       | time=8.42 s | len=11
  n= 45 | status=ok       | time=31.59 s | len=12
  n= 50 | status=ok       | time=71.30 s | len=12
  n= 55 | status=ok       | time=119.53 s | len=12
  n= 60 | status=ok       | time=236.25 s | len=12
  n= 65 | status=ok       | time=386.61 s | len=12
  n= 70 | status=ok       | time=531.45 s | len=12
  n= 75 | status=ok       | time=1054.63 s | len=12
  n= 80 | status=ok       | time=1645.25 s | len=13
  n= 85 | status=deadline | time=1800.00 s | len=13
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_incr

,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,35,CARBOPLATIN > CHEMBL:CHEMBL493863 > FULVESTRAN...,4,0.008579,ok
1,TA-RP-PATH,10,35,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > CA...,7,0.043250,ok
2,TA-RP-PATH,15,35,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > CA...,7,0.085211,ok
3,TA-RP-PATH,20,35,SULFASALAZINE > CYCLOPHOSPHAMIDE ANHYDROUS > D...,8,0.187121,ok
4,TA-RP-PATH,25,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > CYCLOPHO...,9,0.416068,ok
5,TA-RP-PATH,30,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > MK-2461 ...,9,0.863952,ok
6,TA-RP-PATH,35,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > CYCLOPHO...,10,1.746713,ok
7,TA-RP-PATH,40,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SODIUM ...,11,8.424801,ok
8,TA-RP-PATH,45,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,12,31.589883,ok
9,TA-RP-PATH,50,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,12,71.299621,ok


In [ ]:
tarp_drug_increasing("efficacy-diabities.csv",
                      lam_fixed=25,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_diabities_l25.xlsx")

TA-RP-PATH drug-sweep at λ=25; total drugs = 273 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=5
  n= 10 | status=ok       | time=0.03 s | len=8
  n= 15 | status=ok       | time=0.09 s | len=8
  n= 20 | status=ok       | time=0.24 s | len=9
  n= 25 | status=ok       | time=0.94 s | len=11
  n= 30 | status=ok       | time=4.27 s | len=12
  n= 35 | status=ok       | time=10.63 s | len=12
  n= 40 | status=ok       | time=62.08 s | len=13
  n= 45 | status=ok       | time=316.20 s | len=14
  n= 50 | status=ok       | time=791.41 s | len=14
  n= 55 | status=ok       | time=1554.69 s | len=14
  n= 60 | status=deadline | time=1800.00 s | len=14
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_increasing30m_diabities_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,25,CARBOPLATIN > CHEMBL:CHEMBL493863 > LEUCOVORIN...,5,0.008324,ok
1,TA-RP-PATH,10,25,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > WO...,8,0.034602,ok
2,TA-RP-PATH,15,25,CYCLOPHOSPHAMIDE ANHYDROUS > MIDAZOLAM HYDROCH...,8,0.092290,ok
3,TA-RP-PATH,20,25,SULFASALAZINE > OXIDOPAMINE HYDROCHLORIDE > CA...,9,0.236717,ok
4,TA-RP-PATH,25,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > PREGNANT...,11,0.939347,ok
5,TA-RP-PATH,30,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,12,4.269166,ok
6,TA-RP-PATH,35,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > MK-2461 ...,12,10.630290,ok
7,TA-RP-PATH,40,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SODIUM ...,13,62.081259,ok
8,TA-RP-PATH,45,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,14,316.201117,ok
9,TA-RP-PATH,50,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > NIFEDIP...,14,791.414300,ok


In [4]:
tarp_drug_increasing("efficacy-bpco.csv",
                      lam_fixed=25,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_bpco_l25.xlsx")

TA-RP-PATH drug-sweep at λ=25; total drugs = 238 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.05 s | len=6
  n= 15 | status=ok       | time=0.10 s | len=6
  n= 20 | status=ok       | time=0.25 s | len=8
  n= 25 | status=ok       | time=0.57 s | len=9
  n= 30 | status=ok       | time=1.77 s | len=10
  n= 35 | status=ok       | time=7.43 s | len=12
  n= 40 | status=ok       | time=15.44 s | len=12
  n= 45 | status=ok       | time=56.36 s | len=13
  n= 50 | status=ok       | time=213.87 s | len=14
  n= 55 | status=ok       | time=366.81 s | len=14
  n= 60 | status=ok       | time=1427.09 s | len=15
  n= 65 | status=deadline | time=1800.00 s | len=16
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_increasing30m_bpco_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,25,OSIMERTINIB > FULVESTRANT > NICOTINE POLACRILEX,3,0.014288,ok
1,TA-RP-PATH,10,25,THEOPHYLLINE > SORAFENIB > FULVESTRANT > ALPHA...,6,0.045879,ok
2,TA-RP-PATH,15,25,THEOPHYLLINE > OLANZAPINE > INTERLEUKIN-11 > A...,6,0.101304,ok
3,TA-RP-PATH,20,25,THEOPHYLLINE > OLANZAPINE > DICLOFENAC SODIUM ...,8,0.252415,ok
4,TA-RP-PATH,25,25,THEOPHYLLINE > DICLOFENAC SODIUM > SORAFENIB >...,9,0.574359,ok
5,TA-RP-PATH,30,25,THEOPHYLLINE > DICLOFENAC SODIUM > CARBAMAZEPI...,10,1.767309,ok
6,TA-RP-PATH,35,25,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,12,7.426254,ok
7,TA-RP-PATH,40,25,PAXALISIB > INSULIN > DICLOFENAC SODIUM > VORI...,12,15.439289,ok
8,TA-RP-PATH,45,25,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,13,56.359100,ok
9,TA-RP-PATH,50,25,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,14,213.874374,ok


In [5]:
tarp_drug_increasing("efficacy-bpco.csv",
                      lam_fixed=30,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_bpco_l30.xlsx")

TA-RP-PATH drug-sweep at λ=30; total drugs = 238 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.03 s | len=5
  n= 15 | status=ok       | time=0.08 s | len=6
  n= 20 | status=ok       | time=0.16 s | len=7
  n= 25 | status=ok       | time=0.28 s | len=8
  n= 30 | status=ok       | time=0.57 s | len=10
  n= 35 | status=ok       | time=1.19 s | len=11
  n= 40 | status=ok       | time=2.12 s | len=11
  n= 45 | status=ok       | time=8.90 s | len=12
  n= 50 | status=ok       | time=29.90 s | len=13
  n= 55 | status=ok       | time=42.45 s | len=13
  n= 60 | status=ok       | time=151.83 s | len=14
  n= 65 | status=ok       | time=280.25 s | len=15
  n= 70 | status=ok       | time=688.03 s | len=16
  n= 75 | status=ok       | time=1504.92 s | len=16
  n= 80 | status=deadline | time=1800.00 s | len=16
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_increasing30m_bpco_l30.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,30,OSIMERTINIB > FULVESTRANT > NICOTINE POLACRILEX,3,0.007860,ok
1,TA-RP-PATH,10,30,THEOPHYLLINE > SORAFENIB > ALPHA-TOCOPHEROL > ...,5,0.032466,ok
2,TA-RP-PATH,15,30,OLANZAPINE > SORAFENIB > INTERLEUKIN-11 > ALPH...,6,0.076241,ok
3,TA-RP-PATH,20,30,THEOPHYLLINE > DICLOFENAC SODIUM > INTERLEUKIN...,7,0.155188,ok
4,TA-RP-PATH,25,30,THEOPHYLLINE > DICLOFENAC SODIUM > CARBAMAZEPI...,8,0.282529,ok
5,TA-RP-PATH,30,30,PAXALISIB > CAPECITABINE > SORAFENIB > CARBAMA...,10,0.571799,ok
6,TA-RP-PATH,35,30,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,11,1.187933,ok
7,TA-RP-PATH,40,30,PAXALISIB > INSULIN > DICLOFENAC SODIUM > VORI...,11,2.116908,ok
8,TA-RP-PATH,45,30,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,12,8.904439,ok
9,TA-RP-PATH,50,30,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,13,29.903503,ok


In [6]:
tarp_drug_increasing("efficacy-bpco.csv",
                      lam_fixed=35,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_bpco_l35.xlsx")

TA-RP-PATH drug-sweep at λ=35; total drugs = 238 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.03 s | len=5
  n= 15 | status=ok       | time=0.08 s | len=5
  n= 20 | status=ok       | time=0.17 s | len=7
  n= 25 | status=ok       | time=0.24 s | len=8
  n= 30 | status=ok       | time=0.43 s | len=9
  n= 35 | status=ok       | time=0.76 s | len=10
  n= 40 | status=ok       | time=1.20 s | len=10
  n= 45 | status=ok       | time=4.35 s | len=11
  n= 50 | status=ok       | time=9.20 s | len=12
  n= 55 | status=ok       | time=13.96 s | len=12
  n= 60 | status=ok       | time=44.10 s | len=13
  n= 65 | status=ok       | time=77.85 s | len=14
  n= 70 | status=ok       | time=180.39 s | len=15
  n= 75 | status=ok       | time=357.66 s | len=15
  n= 80 | status=ok       | time=637.11 s | len=15
  n= 85 | status=ok       | time=1073.92 s | len=15
  n= 90 | status=deadline | time=1800.00 s | len=15
  ⏱️ Soft deadline hit; st

,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,35,OSIMERTINIB > FULVESTRANT > NICOTINE POLACRILEX,3,0.007694,ok
1,TA-RP-PATH,10,35,THEOPHYLLINE > SORAFENIB > ALPHA-TOCOPHEROL > ...,5,0.032667,ok
2,TA-RP-PATH,15,35,OSIMERTINIB > INTERLEUKIN-11 > ALPHA-TOCOPHERO...,5,0.076340,ok
3,TA-RP-PATH,20,35,THEOPHYLLINE > DICLOFENAC SODIUM > INTERLEUKIN...,7,0.165071,ok
4,TA-RP-PATH,25,35,THEOPHYLLINE > DICLOFENAC SODIUM > CARBAMAZEPI...,8,0.241524,ok
5,TA-RP-PATH,30,35,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,9,0.433941,ok
6,TA-RP-PATH,35,35,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,10,0.759611,ok
7,TA-RP-PATH,40,35,PAXALISIB > CAPECITABINE > DICLOFENAC SODIUM >...,10,1.200646,ok
8,TA-RP-PATH,45,35,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,11,4.351225,ok
9,TA-RP-PATH,50,35,CLOZAPINE > PAXALISIB > NIFEDIPINE > RESVERATR...,12,9.201196,ok


In [4]:
tarp_drug_increasing("efficacy-diabetes.csv",
                      lam_fixed=35,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_diab_l35.xlsx")

TA-RP-PATH drug-sweep at λ=35; total drugs = 171 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.04 s | len=6
  n= 15 | status=ok       | time=0.10 s | len=8
  n= 20 | status=ok       | time=0.22 s | len=8
  n= 25 | status=ok       | time=0.41 s | len=8
  n= 30 | status=ok       | time=1.12 s | len=9
  n= 35 | status=ok       | time=3.53 s | len=9
  n= 40 | status=ok       | time=3.62 s | len=9
  n= 45 | status=ok       | time=9.90 s | len=10
  n= 50 | status=ok       | time=16.74 s | len=10
  n= 55 | status=ok       | time=25.77 s | len=10
  n= 60 | status=ok       | time=51.22 s | len=10
  n= 65 | status=ok       | time=89.07 s | len=11
  n= 70 | status=ok       | time=148.12 s | len=11
  n= 75 | status=ok       | time=229.03 s | len=12
  n= 80 | status=ok       | time=302.30 s | len=12
  n= 85 | status=ok       | time=429.88 s | len=12
  n= 90 | status=ok       | time=1152.21 s | len=13
  n= 95 | status=ok       | 

,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,35,HUMAN > GSK-269962A > BROMOCRIPTINE,3,0.012424,ok
1,TA-RP-PATH,10,35,ATENOLOL > HUMAN > GSK-269962A > ANTISENSE OLI...,6,0.037098,ok
2,TA-RP-PATH,15,35,ASPIRIN > ATENOLOL > GSK-269962A > ANTISENSE O...,8,0.097162,ok
3,TA-RP-PATH,20,35,ASPIRIN > ATENOLOL > GSK-269962A > ANTISENSE O...,8,0.224220,ok
4,TA-RP-PATH,25,35,ASPIRIN > DOXEPIN HYDROCHLORIDE > ISOPROTERENO...,8,0.407262,ok
5,TA-RP-PATH,30,35,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOVITINIB > ...,9,1.116530,ok
6,TA-RP-PATH,35,35,ASPIRIN > DOXORUBICIN HYDROCHLORIDE > RG-1530 ...,9,3.532990,ok
7,TA-RP-PATH,40,35,ASPIRIN > DOXORUBICIN HYDROCHLORIDE > BORTEZOM...,9,3.615227,ok
8,TA-RP-PATH,45,35,ASPIRIN > DOXEPIN HYDROCHLORIDE > METHOTREXATE...,10,9.901528,ok
9,TA-RP-PATH,50,35,ASPIRIN > DOXEPIN HYDROCHLORIDE > QUERCETIN > ...,10,16.743781,ok


In [5]:
tarp_drug_increasing("efficacy-diabetes.csv",
                      lam_fixed=30,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_diab_l30.xlsx")

TA-RP-PATH drug-sweep at λ=30; total drugs = 171 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=3
  n= 10 | status=ok       | time=0.04 s | len=6
  n= 15 | status=ok       | time=0.10 s | len=8
  n= 20 | status=ok       | time=0.22 s | len=8
  n= 25 | status=ok       | time=0.49 s | len=9
  n= 30 | status=ok       | time=1.37 s | len=10
  n= 35 | status=ok       | time=3.79 s | len=10
  n= 40 | status=ok       | time=9.82 s | len=10
  n= 45 | status=ok       | time=24.07 s | len=11
  n= 50 | status=ok       | time=57.00 s | len=11
  n= 55 | status=ok       | time=101.47 s | len=11
  n= 60 | status=ok       | time=224.06 s | len=11
  n= 65 | status=ok       | time=423.06 s | len=12
  n= 70 | status=ok       | time=767.11 s | len=12
  n= 75 | status=ok       | time=1215.49 s | len=12
  n= 80 | status=ok       | time=1613.46 s | len=12
  n= 85 | status=deadline | time=1800.00 s | len=12
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_inc

,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,30,HUMAN > GSK-269962A > BROMOCRIPTINE,3,0.007969,ok
1,TA-RP-PATH,10,30,ATENOLOL > HUMAN > RESVERATROL > ANTISENSE OLI...,6,0.035674,ok
2,TA-RP-PATH,15,30,ASPIRIN > ATENOLOL > ADALIMUMAB-ADBM > ANTISEN...,8,0.104615,ok
3,TA-RP-PATH,20,30,ASPIRIN > ATENOLOL > CYCLOPHOSPHAMIDE ANHYDROU...,8,0.217436,ok
4,TA-RP-PATH,25,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > ISOPROTERENO...,9,0.492532,ok
5,TA-RP-PATH,30,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOVITINIB > ...,10,1.370893,ok
6,TA-RP-PATH,35,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOXORUBICIN ...,10,3.791890,ok
7,TA-RP-PATH,40,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > BORTEZOMIB >...,10,9.819331,ok
8,TA-RP-PATH,45,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > METHOTREXATE...,11,24.065491,ok
9,TA-RP-PATH,50,30,ASPIRIN > DOXEPIN HYDROCHLORIDE > QUERCETIN > ...,11,57.002000,ok


In [6]:
tarp_drug_increasing("efficacy-diabetes.csv",
                      lam_fixed=25,
                      start_size=5, step=5, seed=42,
                      time_budget_s=1800,
                      out_xlsx="tarp_drug_increasing30m_diab_l25.xlsx")

TA-RP-PATH drug-sweep at λ=25; total drugs = 171 (soft deadline per size = 30 min)
  n=  5 | status=ok       | time=0.01 s | len=4
  n= 10 | status=ok       | time=0.06 s | len=6
  n= 15 | status=ok       | time=0.19 s | len=9
  n= 20 | status=ok       | time=0.34 s | len=9
  n= 25 | status=ok       | time=0.80 s | len=10
  n= 30 | status=ok       | time=2.73 s | len=11
  n= 35 | status=ok       | time=9.93 s | len=11
  n= 40 | status=ok       | time=24.04 s | len=11
  n= 45 | status=ok       | time=67.66 s | len=12
  n= 50 | status=ok       | time=179.62 s | len=12
  n= 55 | status=ok       | time=337.85 s | len=12
  n= 60 | status=ok       | time=787.39 s | len=13
  n= 65 | status=ok       | time=1655.04 s | len=13
  n= 70 | status=deadline | time=1800.00 s | len=13
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tarp_drug_increasing30m_diab_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-RP-PATH,5,25,ATENOLOL > HUMAN > GSK-269962A > BROMOCRIPTINE,4,0.011646,ok
1,TA-RP-PATH,10,25,HUMAN > GSK-269962A > ANTISENSE OLIGONUCLEOTID...,6,0.061195,ok
2,TA-RP-PATH,15,25,ASPIRIN > ATENOLOL > GSK-269962A > BROMOCRIPTI...,9,0.185767,ok
3,TA-RP-PATH,20,25,ASPIRIN > ATENOLOL > CYCLOPHOSPHAMIDE ANHYDROU...,9,0.340995,ok
4,TA-RP-PATH,25,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > ISOPROTERENO...,10,0.803911,ok
5,TA-RP-PATH,30,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOVITINIB > ...,11,2.725712,ok
6,TA-RP-PATH,35,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > DOXORUBICIN ...,11,9.927013,ok
7,TA-RP-PATH,40,25,ASPIRIN > DOXORUBICIN HYDROCHLORIDE > BORTEZOM...,11,24.035537,ok
8,TA-RP-PATH,45,25,ASPIRIN > ATENOLOL > METHOTREXATE > BORTEZOMIB...,12,67.659309,ok
9,TA-RP-PATH,50,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > QUERCETIN > ...,12,179.622772,ok
